# 00 · Setup check

Run this notebook **before Session 3** (*Run → Run All Cells*). Every line should show ✅.
If something fails, the last cell prints what to send to the teacher.

How to install the environment: see [`setup/README.md`](../setup/README.md).

In [ ]:
import platform
import sys

print("Python :", sys.version.split()[0])
print("Kernel :", sys.executable)
print("System :", platform.platform())

in_course_env = any(k in sys.executable for k in (".venv", "ub-bigdata-viz", "/opt/venv"))
print("✅ Running inside the course environment" if in_course_env else
      "⚠️  This kernel does not look like the course environment (.venv or ub-bigdata-viz).\n"
      "    Choose it in the kernel menu (top right) and run the notebook again.")
print("✅ Python 3.12 or newer" if sys.version_info >= (3, 12) else "❌ Python is older than 3.12")

## 1. Packages and versions

In [ ]:
from importlib.metadata import PackageNotFoundError, version

REQUIRED = {
    "pandas": "3.0", "numpy": "2.1", "matplotlib": "3.10", "seaborn": "0.13", "scipy": "1.14",
    "geopandas": "1.1", "pyogrio": "0.10", "shapely": "2.1", "pyarrow": "18", "plotly": "6",
    "bokeh": "3.6", "openpyxl": "3.1", "requests": "2.32", "python-dotenv": "1.0", "sodapy": "2.2",
}


def as_tuple(v):
    return tuple(int(p) for p in v.split(".")[:2] if p.isdigit())


problems = [] if sys.version_info >= (3, 12) else ["Python < 3.12"]
for pkg, minimum in REQUIRED.items():
    try:
        v = version(pkg)
    except PackageNotFoundError:
        problems.append(f"{pkg} is not installed")
        print(f"❌ {pkg:14s} missing")
        continue
    ok = as_tuple(v) >= as_tuple(minimum)
    print(f"{'✅' if ok else '❌'} {pkg:14s} {v:10s} (needs ≥ {minimum})")
    if not ok:
        problems.append(f"{pkg} {v} < {minimum}")

## 2. Does everything work together?

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({"city": ["Barcelona", "Girona"], "pop": [1.66, 0.10]})
print("pandas string dtype:", df["city"].dtype, "(pandas 3 shows 'str')")

comarques = gpd.read_file(DATA / "icgc/divisions-administratives-v2r1-comarques-250000-20250730.json")
print(f"✅ GeoPandas read {len(comarques)} comarques (CRS {comarques.crs})")

fig, ax = plt.subplots(figsize=(4, 4))
comarques.plot(ax=ax, color="lightgrey", edgecolor="white")
ax.set_axis_off()
ax.set_title("If you see Catalonia, plotting works")
plt.show()

## 3. Course data

In [ ]:
expected = [
    "imdb/imdb_top_1000.csv",
    "examples/energy_consumption_municipalities.parquet",
    "examples/covid_vaccination_daily.parquet",
    "examples/bcn_accidents_2024.parquet",
    "icgc/divisions-administratives-v2r1-municipis-1000000-20250730.json",
    "bcn/barris.geojson",
]
for rel in expected:
    ok = (DATA / rel).exists()
    print(f"{'✅' if ok else '❌'} data/{rel}")
    if not ok:
        problems.append(f"missing data/{rel}")

## 4. Internet access to the open data portal (optional)

In [ ]:
import os

import requests

if os.environ.get("OFFLINE") == "1":
    print("Skipped (OFFLINE=1)")
else:
    try:
        r = requests.get("https://analisi.transparenciacatalunya.cat/resource/8idm-becu.json",
                         params={"$limit": 1}, timeout=15)
        print("✅ Transparència Catalunya API reachable" if r.ok else f"⚠️  API answered {r.status_code}")
    except requests.RequestException as e:
        print("⚠️  No access to the API (fine offline):", type(e).__name__)

## 5. Result

In [ ]:
if problems:
    print("❌ Something is not right. Send this to the teacher:\n")
    print("Python", sys.version.split()[0], "|", sys.executable, "|", platform.platform())
    for p in problems:
        print(" -", p)
else:
    print("🎉 Your environment is ready for the course.")